In [18]:
from arcgis.gis import GIS
from arcgis.features import GeoAccessor, GeoSeriesAccessor
import geopandas as gpd
import pandas as pd

# Connect to ArcGIS Online
gis = GIS("home")

# Retrieve the existing feature service layer
boem_samplesites = gis.content.get('29407067efe9494e9d5c49b7afbbdccb').layers[0]
boem_samplesites_dbf = pd.DataFrame(boem_samplesites.query().sdf)



# Load the new dataset
# For a shapefile
new_data_gdf = gpd.read_file(r'C:\Users\Mike.Bollinger\Downloads\SamplingExamples\SRS.shp')
new_data_dbf = pd.DataFrame(new_data_gdf.query().sdf)
#new_data_gdf = new_data_gdf.rename(columns={'FID': 'fid', 'geometry': 'SHAPE'})
# For a GeoPackage
#new_data_gdf = gpd.read_file(r"D:\BOEM_CarolinaLongBay_GIS\SRS.gpkg", layer='main.SRS')
new_data_dbf.head()

TypeError: query() missing 1 required positional argument: 'expr'

In [17]:



# Assuming we need to keep only relevant columns from boem_samplesites_sdf
# and append new_data_gdf with necessary adjustments.
# If necessary, rename columns in new_data_gdf to match boem_samplesites_sdf
# For example:
# new_data_gdf = new_data_gdf.rename(columns={'old_name': 'new_name'})

# Select the relevant columns from new_data_gdf that match the feature layer schema
# Assuming new_data_gdf has only one field and we need to add it to boem_samplesites_sdf
# Here, we add a placeholder for the missing fields
for col in boem_samplesites_dbf.columns:
    if col not in new_data_gdf.columns:
        new_data_gdf[col] = None


# Assuming the shapefile is projected points without CRS, set the CRS using the one from boem_samplesites_dbf
new_data_gdf.crs = boem_samplesites_dbf['SHAPE'].spatialReference['wkid']


# Inspect the columns to make sure they match
print(boem_samplesites_dbf.columns)
print(new_data_gdf.columns)


ValueError: Address column not found in dataframe

In [12]:


# Combine the existing feature service layer data with new data
combined_dbf = boem_samplesites_dbf.append(new_data_gdf, ignore_index=True)

combined_dbf = pd.concat([boem_samplesites_dbf, new_data_gdf], ignore_index=True)

combined_dbf.head()

C:\Users\Mike.Bollinger\AppData\Local\Temp\1\ipykernel_13952\2160648998.py:2: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  combined_dbf = boem_samplesites_dbf.append(new_data_gdf, ignore_index=True)


,fid,AUV_Completed_Date,ROV_Completed_Date,DropCam_Completed_Date,Most_Recent_Survey_Date,Status,SHAPE
0,1,NaN,NaN,NaN,NaN,NaN,"{'x': -8672266.923696559, 'y': 3956340.6028946..."
1,2,NaN,NaN,NaN,NaN,NaN,"{'x': -8671286.636925546, 'y': 3964367.1259985..."
2,3,NaN,NaN,NaN,NaN,NaN,"{'x': -8670023.981887085, 'y': 3959888.8789360..."
3,4,NaN,NaN,NaN,NaN,NaN,"{'x': -8667563.429121977, 'y': 3964145.6021091..."
4,5,NaN,NaN,NaN,NaN,NaN,"{'x': -8667673.16680463, 'y': 3962770.13238120..."


In [17]:

# Convert combined data to a spatially enabled DataFrame
combined_sdf = GeoAccessor.from_dataframe(combined_sdf)

# Overwrite the existing feature layer with the updated data
boem_samplesites.edit_features(adds=combined_sdf.to_dict(orient='records'))

In [18]:
# Ensure both GeoDataFrames have the same CRS
if new_data_gdf.crs != f"EPSG:{epsg_code}":
    new_data_gdf = new_data_gdf.to_crs(epsg_code)

# Convert the existing DataFrame to GeoDataFrame
boem_samplesites_gdf = gpd.GeoDataFrame(boem_samplesites_dbf, geometry='geometry', crs=f"EPSG:{epsg_code}")

# Concatenate the new data to the existing data
combined_data = pd.concat([boem_samplesites_gdf, new_data_gdf], ignore_index=True)

# Ensure combined data is a GeoDataFrame with correct CRS
combined_gdf = gpd.GeoDataFrame(combined_data, geometry='geometry', crs=f"EPSG:{epsg_code}")

# Convert GeoDataFrame to FeatureSet
combined_features = combined_gdf.spatial.to_featureset()

# Update the feature layer
boem_samplesites.edit_features(updates=combined_features)

ValueError: Unknown column geometry

In [20]:
from arcgis.gis import GIS
from arcgis.features import GeoAccessor, GeoSeriesAccessor
import geopandas as gpd
import pandas as pd

# Connect to ArcGIS Online
gis = GIS("home")

# Retrieve the existing feature service layer
boem_samplesites = gis.content.get('29407067efe9494e9d5c49b7afbbdccb').layers[0]

# Load the existing data into a GeoDataFrame
boem_samplesites_gdf = boem_samplesites.query().sdf

# Check if the geometry column is correctly set
if 'geometry' not in boem_samplesites_gdf.columns:
    boem_samplesites_gdf = boem_samplesites_gdf.set_geometry('SHAPE')

# Inspect the properties to find the spatial reference
print(boem_samplesites.properties)

# Extract spatial reference
spatial_ref = boem_samplesites.properties.get('spatialReference', None)
if not spatial_ref:
    spatial_ref = boem_samplesites.properties['extent']['spatialReference']

# Determine the correct EPSG code
epsg_code = spatial_ref.get('latestWkid', spatial_ref.get('wkid', None))

# Ensure the existing GeoDataFrame has the correct CRS
boem_samplesites_gdf.crs = f"EPSG:{epsg_code}"
boem_samplesites_gdf = boem_samplesites_gdf.to_crs(epsg_code)

# Load the new dataset from the GeoPackage
new_data_gdf = gpd.read_file(r"D:\BOEM_CarolinaLongBay_GIS\SRS.gpkg")

# Set the CRS for the new GeoDataFrame if not already set
if new_data_gdf.crs is None:
    new_data_gdf.crs = 'EPSG:26917'  # Assuming the GeoPackage's CRS is EPSG:26917

# Ensure the new GeoDataFrame has the same CRS
if new_data_gdf.crs != f"EPSG:{epsg_code}":
    new_data_gdf = new_data_gdf.to_crs(epsg_code)

# Concatenate the new data to the existing data
combined_gdf = pd.concat([boem_samplesites_gdf, new_data_gdf], ignore_index=True)

# Ensure combined data is a GeoDataFrame with correct CRS
combined_gdf = gpd.GeoDataFrame(combined_gdf, geometry='geometry', crs=f"EPSG:{epsg_code}")

# Convert GeoDataFrame to FeatureSet
combined_features = combined_gdf.spatial.to_featureset()

# Update the feature layer
boem_samplesites.edit_features(updates=combined_features)



{
  "currentVersion": 11.2,
  "id": 0,
  "name": "main.SRS",
  "type": "Feature Layer",
  "serviceItemId": "29407067efe9494e9d5c49b7afbbdccb",
  "cacheMaxAge": 30,
  "displayField": "",
  "description": "",
  "copyrightText": "",
  "defaultVisibility": true,
  "editingInfo": {
    "lastEditDate": 1716490679989,
    "schemaLastEditDate": 1716490677354,
    "dataLastEditDate": 1716470860573
  },
  "relationships": [],
  "isDataVersioned": false,
  "hasContingentValuesDefinition": false,
  "supportsAppend": true,
  "supportsCalculate": true,
  "supportsASyncCalculate": true,
  "supportsTruncate": true,
  "supportsAttachmentsByUploadId": true,
  "supportsAttachmentsResizing": true,
  "supportsRollbackOnFailureParameter": true,
  "supportsStatistics": true,
  "supportsExceedsLimitStatistics": true,
  "supportsAdvancedQueries": true,
  "supportsValidateSql": true,
  "supportsCoordinatesQuantization": true,
  "supportsLayerOverrides": true,
  "supportsTilesAndBasicQueriesMode": true,
  "suppo

KeyError: <class 'shapely.geometry.point.Point'>